In [11]:
pip install pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pinecone import Pinecone, ServerlessSpec

In [3]:
documents = [
    {
        "id": "doc-001",
        "text": "Pinecone is a fully managed vector database for search and recommendation.",
        "category": "documentation",
        "tag": "pinecone",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-intro"
    },
    {
        "id": "doc-002",
        "text": "To use Pinecone with Python, you create an index and upsert vectors with metadata.",
        "category": "documentation",
        "tag": "python",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-python"
    },
    {
        "id": "doc-003",
        "text": "Vector databases store embeddings that capture semantic meaning for semantic search.",
        "category": "blog",
        "tag": "vector-db",
        "difficulty": "intermediate",
        "url": "https://example.com/vector-db-concepts"
    },
    {
        "id": "doc-004",
        "text": "You can filter Pinecone search results using metadata such as category or difficulty.",
        "category": "faq",
        "tag": "metadata",
        "difficulty": "beginner",
        "url": "https://example.com/pinecone-metadata"
    },
    {
        "id": "doc-005",
        "text": "In Retrieval-Augmented Generation, a vector database like Pinecone stores document chunks.",
        "category": "blog",
        "tag": "rag",
        "difficulty": "intermediate",
        "url": "https://example.com/rag-pinecone"
    }
]


In [4]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # .env sits in the repo root, one level above this notebook

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

pc = Pinecone(api_key=PINECONE_API_KEY)

In [5]:
import os
import requests
import numpy as np
from dotenv import load_dotenv, find_dotenv
from typing import Union, List

load_dotenv(find_dotenv(usecwd=True))
EURI_API_KEY = os.environ["EURI_API_KEY"]

def generate_embeddings(texts: Union[str, List[str]]):
    # Always make the input a list
    if isinstance(texts, str):
        texts = [texts]

    url = "https://api.euron.one/api/v1/euri/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {EURI_API_KEY}"
    }
    payload = {
        "input": texts,
        "model": "text-embedding-3-small"
    }

    response = requests.post(url, headers=headers, json=payload)
    data = response.json()

    # Convert each embedding to numpy array
    embeddings = [np.array(item["embedding"], dtype=np.float32) for item in data["data"]]

    # Return single vector OR batch of vectors
    return embeddings[0] if len(embeddings) == 1 else np.stack(embeddings)

In [6]:
INDEX_NAME = "euron-pinecone-euri-demo"

In [7]:
pc.list_indexes() # Check if the index already exists

IndexList([<name='euron-pinecone-euri-demo', dim=1536, ready=True>])

In [8]:
pc.delete_index(INDEX_NAME)

In [9]:
# Creating a new index in the our Pinecone environment.
pc.create_index(
    name = INDEX_NAME, # Name of the pinecone index which we have already specified in the INDEX_NAME variable above.
    dimension = 1536, # dimension means the number of features in the vector representation of the data. In this case, each vector will have 1536 dimensions.
    metric = "cosine", # metric specifies the distance metric used to compare vectors in the index. "cosine" means that the similarity between vectors will be measured using cosine similarity, which is a common choice for text embeddings.
    spec = ServerlessSpec(  # serverless spec is a configuration for the serverless deployment of the index, specifying the cloud provider and region where the index will be hosted.
        cloud = "aws",
        region = "us-east-1"
        )
)

IndexModel(name='euron-pinecone-euri-demo', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://euron-pinecone-euri-demo-zapfl1c.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [10]:
documents

[{'id': 'doc-001',
  'text': 'Pinecone is a fully managed vector database for search and recommendation.',
  'category': 'documentation',
  'tag': 'pinecone',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-intro'},
 {'id': 'doc-002',
  'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
  'category': 'documentation',
  'tag': 'python',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-python'},
 {'id': 'doc-003',
  'text': 'Vector databases store embeddings that capture semantic meaning for semantic search.',
  'category': 'blog',
  'tag': 'vector-db',
  'difficulty': 'intermediate',
  'url': 'https://example.com/vector-db-concepts'},
 {'id': 'doc-004',
  'text': 'You can filter Pinecone search results using metadata such as category or difficulty.',
  'category': 'faq',
  'tag': 'metadata',
  'difficulty': 'beginner',
  'url': 'https://example.com/pinecone-metadata'},
 {'id': 'doc-005',
  'text': 'In Retrie

In [11]:
texts = [doc["text"] for doc in documents] # This line of code extracts the 'text' field from each document in the 'documents' list and creates a new list called 'texts' that contains only the text content of each document.

In [12]:
texts

['Pinecone is a fully managed vector database for search and recommendation.',
 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
 'Vector databases store embeddings that capture semantic meaning for semantic search.',
 'You can filter Pinecone search results using metadata such as category or difficulty.',
 'In Retrieval-Augmented Generation, a vector database like Pinecone stores document chunks.']

In [13]:
# Generate embeddings for the texts using the EURI API
doc_embeddings = generate_embeddings(texts)

In [14]:
doc_embeddings

array([[-0.00132942,  0.00098705,  0.02589417, ...,  0.0103302 ,
        -0.00441742, -0.00444794],
       [ 0.02902222,  0.02787781,  0.04516602, ..., -0.01396942,
         0.004776  ,  0.01304626],
       [-0.02900696,  0.02354431,  0.01060486, ...,  0.0275116 ,
         0.00490189,  0.00447464],
       [ 0.02288818, -0.01055908,  0.05374146, ..., -0.00687027,
         0.02050781, -0.01083374],
       [ 0.01342773,  0.02363586,  0.0447998 , ..., -0.00083542,
         0.04611206,  0.01820374]], shape=(5, 1536), dtype=float32)

In [15]:
documents, doc_embeddings

([{'id': 'doc-001',
   'text': 'Pinecone is a fully managed vector database for search and recommendation.',
   'category': 'documentation',
   'tag': 'pinecone',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-intro'},
  {'id': 'doc-002',
   'text': 'To use Pinecone with Python, you create an index and upsert vectors with metadata.',
   'category': 'documentation',
   'tag': 'python',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-python'},
  {'id': 'doc-003',
   'text': 'Vector databases store embeddings that capture semantic meaning for semantic search.',
   'category': 'blog',
   'tag': 'vector-db',
   'difficulty': 'intermediate',
   'url': 'https://example.com/vector-db-concepts'},
  {'id': 'doc-004',
   'text': 'You can filter Pinecone search results using metadata such as category or difficulty.',
   'category': 'faq',
   'tag': 'metadata',
   'difficulty': 'beginner',
   'url': 'https://example.com/pinecone-metadata'},
  {'id': 'doc-0

In [16]:
# What is the relationship between the documents and their embeddings? Each document has a corresponding embedding that 
# captures its semantic meaning. The embeddings are numerical representations of the text content of the documents, 
# allowing for efficient similarity searches and comparisons in vector space.

# And why are we zipping them together? We are zipping the documents and their corresponding embeddings together to create 
# pairs of (document, embedding). This allows us to easily associate each document with its embedding when we upsert them 
# into the Pinecone index.
zip(documents, doc_embeddings)

In [17]:
# Example of the usage of zip function:

l = [4,7,8]
m = ['a', 'b', 'c']

list(zip(l,m))

[(4, 'a'), (7, 'b'), (8, 'c')]

Before we run the below cell, some points to know:
- Currently, we have only created a pinecone index through python code and we do not have any record/data present in this index, hence we need to upsert our records/data in this pinecone index.
- Our data is present in two separate variables:
```
    1.) 'documents' -> Human-readable english
    2.) 'doc_embeddings' -> Machine-readable(in vectors)
```
- Pinecone cannot accept two distinct groups/variables like the above. Pinecone only requires one distinct list which have the both glued together which we will do next.
- Upsert is a database operation that updates an existing record if it already exists or inserts a new record if it does not.



In [ ]:
# Using the below code, we are upserting the documents and their embeddings into the Pinecone index.

vector_to_upsert = []
for doc,emb in  zip(documents, doc_embeddings):
    metadata = {
        "category": doc["category"],
        "tag": doc["tag"],
        "difficulty": doc["difficulty"],
        "url": doc["url"],
        "text": doc["text"]
        }
    vector_item = {
        "id": doc["id"],
        "values": emb.tolist(),
        "metadata": metadata
    }
    
    vector_to_upsert.append(vector_item)

- Example of how will the vector_to_upsert will look like:
```
vector_to_upsert = [
    {"id": "doc-001", "values": [ 0.021, -0.184, ...], "metadata": {"tag": "pinecone",   "text": "Pinecone is a fully...",  ...}},
    {"id": "doc-002", "values": [-0.077,  0.312, ...], "metadata": {"tag": "python",     "text": "To use Pinecone with...", ...}},
    {"id": "doc-003", "values": [ ...             ], "metadata": {"tag": "vector-db",  "text": "Vector databases store...", ...}},
    {"id": "doc-004", "values": [ ...             ], "metadata": {"tag": "metadata",   "text": "You can filter Pinecone...", ...}},
    {"id": "doc-005", "values": [ ...             ], "metadata": {"tag": "rag",        "text": "In Retrieval-Augmented...", ...}},
]
```

In [ ]:
# Retrieves the first item from the `vector_to_upsert` list, which contains the first document's ID, its corresponding 
# embedding values(converted to a list), and its associated metadata. This allows you to inspect the structure and content of the first vector 
# that will be upserted into the Pinecone index.

vector_to_upsert[0] 

{'id': 'doc-001',
 'values': [-0.0013294219970703125,
  0.0009870529174804688,
  0.0258941650390625,
  0.00646209716796875,
  0.0272979736328125,
  4.273653030395508e-05,
  -0.0019664764404296875,
  0.05804443359375,
  0.008941650390625,
  -0.00664520263671875,
  0.027557373046875,
  -0.00858306884765625,
  0.0205230712890625,
  -0.07861328125,
  0.0141448974609375,
  -0.0181427001953125,
  -0.028839111328125,
  0.0074005126953125,
  0.033294677734375,
  0.0390625,
  0.015777587890625,
  0.060516357421875,
  0.0291290283203125,
  -0.0181732177734375,
  0.0179290771484375,
  0.05926513671875,
  -0.033599853515625,
  0.04156494140625,
  0.08160400390625,
  -0.057525634765625,
  0.0217132568359375,
  -0.0230712890625,
  -0.0245208740234375,
  -0.00848388671875,
  0.0275115966796875,
  -3.0100345611572266e-05,
  -0.003955841064453125,
  0.016143798828125,
  -0.00727081298828125,
  0.023895263671875,
  -0.01788330078125,
  0.044189453125,
  0.0046844482421875,
  -0.0220947265625,
  0.020507

In [21]:
index = pc.Index(INDEX_NAME)

In [22]:
index

Index(host='https://euron-pinecone-euri-demo-zapfl1c.svc.aped-4627-b74a.pinecone.io')

- Pinecone has upsert(update + insert) operation, rather than plain insert or update. This means that if a vector with the same ID already exists in the index, it will be updated with the new values and metadata. If it does not exist, it will be inserted as a new vector.
- In Pinecone, "upserting overwrites the entire record." Fields are not merged. Example:
    - If doc-001 is already in your index with all five metadata fields, and you later upsert:
        ```
        {"id": "doc-001", "values": [...], "metadata": {"category": "blog"}}
        ```
        You don't end up with category changed and the rest intact. You end up with a record whose metadata is only {"category": "blog"} — tag, difficulty, url, and text are gone

- For "change one field, leave everything else alone," Pinecone gives you a different method:
    ```
    index.update(id="doc-001", set_metadata={"difficulty": "advanced"})
    ```

In [ ]:
index.upsert(vectors=vector_to_upsert) # upserting the vectors into the pinecone index.

UpsertResponse(upserted_count=5)

In [ ]:
# testing the functionality of the index by querying it with a sample query and retrieving the top 3 most similar documents based on their 
# embeddings.
query_text = "How to use Pinecone with Python?"
query_embedding = generate_embeddings(query_text)

In [25]:
query_embedding

array([ 0.00062561, -0.03536987,  0.0463562 , ..., -0.01100159,
       -0.01567078,  0.03004456], shape=(1536,), dtype=float32)

In [ ]:
# .tolist() method in Python is used to convert NumPy arrays or Pandas Series/Indexes into native Python lists.
pi_response = index.query(
    vector=query_embedding.tolist(), 
    top_k=3, # Give me the 3 closest matches
    include_metadata=True
)

In [27]:
from pprint import pprint
pprint(pi_response.to_dict())

{'matches': [{'id': 'doc-002',
              'metadata': {'category': 'documentation',
                           'difficulty': 'beginner',
                           'tag': 'python',
                           'text': 'To use Pinecone with Python, you create an '
                                   'index and upsert vectors with metadata.',
                           'url': 'https://example.com/pinecone-python'},
              'score': 0.702976286,
              'sparse_values': None,
              'values': []},
             {'id': 'doc-001',
              'metadata': {'category': 'documentation',
                           'difficulty': 'beginner',
                           'tag': 'pinecone',
                           'text': 'Pinecone is a fully managed vector '
                                   'database for search and recommendation.',
                           'url': 'https://example.com/pinecone-intro'},
              'score': 0.566469729,
              'sparse_values': None

In [ ]:
pi_response_meta = index.query(
    vector=query_embedding.tolist(),
    top_k=3,
    include_metadata=True,
    filter={
        "difficulty": {"$eq": "beginner"}}   # $eq operator means "equal to" and here it is used to filter the results to only include documents where the "difficulty" metadata field is equal to "beginner".
)

In [29]:
from pprint import pprint
pprint(pi_response_meta.to_dict())

{'matches': [{'id': 'doc-002',
              'metadata': {'category': 'documentation',
                           'difficulty': 'beginner',
                           'tag': 'python',
                           'text': 'To use Pinecone with Python, you create an '
                                   'index and upsert vectors with metadata.',
                           'url': 'https://example.com/pinecone-python'},
              'score': 0.702757776,
              'sparse_values': None,
              'values': []},
             {'id': 'doc-001',
              'metadata': {'category': 'documentation',
                           'difficulty': 'beginner',
                           'tag': 'pinecone',
                           'text': 'Pinecone is a fully managed vector '
                                   'database for search and recommendation.',
                           'url': 'https://example.com/pinecone-intro'},
              'score': 0.566704094,
              'sparse_values': None